# Deep learning approach: fine-tuning a pretrained CNN (Keras)

Task: one pediatric chest X-ray in, PNEUMONIA (**Positive**) or NORMAL (**Negative**) out, the same labels as `eval/eval.py`.

## How to run this on Google Colab
1. Put the project folder on Google Drive (default location: `MyDrive/pneumonia_project`). It needs `data/image_index.csv`, `eval/`, `approaches/1_human_baseline/subset_ids.csv`, and **either** the raw images in `data/raw/chest_xray/` **or** the prebuilt cache folder `dl_work/` (smaller, made by `prepare_colab_bundle.py`).
2. In Colab choose **Runtime > Change runtime type > GPU**.
3. Run all cells. Every experiment is cached in Drive, so if Colab disconnects, re-run and it resumes where it stopped.
4. The last cell zips the figures, results and predictions so you can download them into the repo.

Running locally works too. `DL_PROFILE=quick` is a tiny smoke test; the default profile is `standard`.

## What the notebook does (the plan)

| Step | What happens | Why |
|---|---|---|
| 0-1 | Setup; read the project split from `data/image_index.csv`; look at the data | never re-split; see the imbalance |
| 2 | Sanity checks (labels, majority baseline, overfit a tiny batch) | catch bugs cheaply |
| 3 | Model: pretrained CNN + new sigmoid head, two-phase fine-tuning | transfer learning, as the course requires |
| 4 | **Stage 1** capacity (how much to unfreeze), **Stage 2** backbone size, **Stage 3** small random search | tune on validation only |
| 5 | **Stage 4** long training, several seeds, no early stopping | how performance evolves with longer training |
| 6 | Pick the decision threshold on validation (your decision) | prevalence differs between val and test |
| 7 | **Run the test set once** (guarded), save the model and predictions CSV | no peeking while tuning |
| 8 | Metrics with patient-level bootstrap confidence intervals, comparison with human and symbolic | the primary comparison |
| 9 | Figures: ROC/PR/calibration, Grad-CAM, bacterial vs viral, cost and capacity, t-SNE, learning curve, ablations | task performance, failure modes, explainability, fairness, cost |

**Rules of this notebook:** train on `train`, judge on `val`, touch `test` exactly once. Training and inference code uses Python and Keras (course requirement).

## 0. Setup

In [ ]:
import os, sys, json, time, random, copy, hashlib, shutil, subprocess, warnings
os.environ["KERAS_BACKEND"] = "tensorflow"          # Grad-CAM below uses tf.GradientTape
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import tensorflow as tf
import keras
from keras import layers
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.manifold import TSNE
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
tf.get_logger().setLevel("ERROR")          # silence harmless "retracing" warnings from repeated predict() calls

# ------------------------------ settings you may edit ------------------------------
PROFILE = os.environ.get("DL_PROFILE", "standard")    # "quick" (smoke test) | "standard" | "long"
DRIVE_PROJECT = "/content/drive/MyDrive/pneumonia_project"   # Colab only: the project folder on Drive
RERUN = os.environ.get("DL_RERUN", "0") == "1"        # True = ignore cached experiments and retrain
ALLOW_SECOND_TEST_RUN = False                         # the test set is evaluated ONCE; leave False
SEED = 211
# -----------------------------------------------------------------------------------

PROFILES = dict(
    quick=dict(img=96, batch=16, n_train=240, n_val=96, n_test=96, head_epochs=1, ep_sweep=1, fracs=[0.0, 0.5],
               include_scratch=True, backbones=["MobileNetV2", "EfficientNetV2B0"], n_trials=2, ep_tune=1,
               ep_long=2, long_seeds=2, ablation_epochs=1, lc_fracs=[0.5], lc_epochs=1, n_boot=30, tol=0.01),
    standard=dict(img=224, batch=32, n_train=None, n_val=None, n_test=None, head_epochs=3, ep_sweep=12,
                  fracs=[0.0, 0.25, 0.5, 1.0], include_scratch=True,
                  backbones=["MobileNetV2", "EfficientNetV2B0", "ConvNeXtTiny"], n_trials=5, ep_tune=8,
                  ep_long=40, long_seeds=2, ablation_epochs=10, lc_fracs=[0.1, 0.25, 0.5], lc_epochs=10, n_boot=1000, tol=0.01),
    long=dict(img=224, batch=32, n_train=None, n_val=None, n_test=None, head_epochs=3, ep_sweep=20,
              fracs=[0.0, 0.25, 0.5, 1.0], include_scratch=True,
              backbones=["MobileNetV2", "EfficientNetV2B0", "ConvNeXtTiny"], n_trials=6, ep_tune=10,
              ep_long=60, long_seeds=3, ablation_epochs=15, lc_fracs=[0.1, 0.25, 0.5], lc_epochs=15, n_boot=1000, tol=0.01),
)
assert PROFILE in PROFILES, PROFILE
P = PROFILES[PROFILE]
if os.environ.get("DL_BACKBONES"):                     # e.g. DL_BACKBONES=MobileNetV2,EfficientNetV2B0 to skip the large model
    P["backbones"] = os.environ["DL_BACKBONES"].split(",")
MODE = "quick" if PROFILE == "quick" else "full"
IMG, BATCH = P["img"], P["batch"]

# ---- where things live
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path(DRIVE_PROJECT)
else:
    HERE = Path.cwd().resolve()
    PROJECT = next((p for p in [HERE, *HERE.parents] if (p / "data" / "image_index.csv").exists()), HERE)
assert (PROJECT / "data" / "image_index.csv").exists(), f"data/image_index.csv not found under {PROJECT}"

if MODE == "quick":
    WORK = PROJECT / "local" / "dl_quick_work"
    shutil.rmtree(WORK, ignore_errors=True)               # the smoke test always starts clean
    OUT = WORK / "out"
else:
    WORK = PROJECT / ("dl_work" if IN_COLAB else "local/dl_work")
    OUT = PROJECT / "approaches" / "3_deep_learning"
FIG_DIR, RES_DIR, MODEL_DIR, RUN_DIR = OUT / "figures", OUT / "results", OUT / "models", WORK / "runs"
for d in (FIG_DIR, RES_DIR, MODEL_DIR, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- hardware
gpus = tf.config.list_physical_devices("GPU")
DEVICE = (gpus[0].name if gpus else "CPU")
# Mixed precision (float16 maths) roughly doubles speed on a Colab T4; the output layer stays float32.
# If you ever see a dtype error, set the environment variable DL_MIXED=0 (or edit the next line).
MIXED = os.environ.get("DL_MIXED", "auto")
use_mixed = MIXED == "1" or (MIXED == "auto" and bool(gpus) and MODE == "full")
if use_mixed:
    keras.mixed_precision.set_global_policy("mixed_float16")
keras.utils.set_random_seed(SEED)

# ---- plotting defaults (one colour per approach, used everywhere)
C = dict(normal="#4C72B0", pneu="#DD8452", human="#8C8C8C", symbolic="#8172B3", scratch="#55A868", dl="#C44E52")
COL = {"Always positive": C["human"], "Human": C["human"], "Symbolic rule": C["symbolic"],
       "Scratch CNN": C["scratch"], "Deep learning": C["dl"]}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
SAVED = []


def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    SAVED.append(f"figures/{name}.png")
    plt.show()


print(f"profile={PROFILE}  device={DEVICE}  mixed_precision={use_mixed}  tf={tf.__version__}  keras={keras.__version__}  colab={IN_COLAB}")
print(f"project={PROJECT}\nwork={WORK}\noutputs={OUT}")

## 1. Data

`data/image_index.csv` holds the **one project-wide split** (`train` / `val` / `test`), grouped by patient. This notebook only reads it.

- Each X-ray becomes **grayscale**, resized once to a square (224x224) and cached as `uint8` (decoding 1500-pixel JPEGs every epoch would dominate training time).
- The model itself turns it into 3 channels (pretrained models expect colour) and applies **random augmentation** (small rotation, zoom, shift, brightness and contrast) *only while training*; these are Keras layers inside the model and switch off automatically at prediction time.
- **No horizontal flip by default**: the heart sits on one side of the chest, so flipping creates anatomy that never occurs. It appears as an ablation.
- The **file names are never an input**. They contain "bacteria"/"virus", which would leak the label. The subtype is read only afterwards, for the fairness analysis.

In [ ]:
index = pd.read_csv(PROJECT / "data" / "image_index.csv")
# Pneumonia file names carry the subtype (person12_bacteria_34.jpeg). Used ONLY for the post-hoc subgroup analysis.
index["subtype"] = np.where(index.label == 1, np.where(index.path.str.contains("_bacteria_"), "bacterial", "viral"), "normal")


def take(frame, n):
    return frame if n is None or n >= len(frame) else frame.sample(n=n, random_state=SEED)


df = pd.concat([take(index[index.split == s], P[f"n_{s}"]) for s in ("train", "val", "test")], ignore_index=True)
idx = {s: np.where(df.split.values == s)[0] for s in ("train", "val", "test")}      # row positions per split
y = df.label.values.astype(np.float32)


def load_gray(rel):
    im = Image.open(PROJECT / rel).convert("L").resize((IMG, IMG), Image.Resampling.LANCZOS)
    return np.asarray(im, dtype=np.uint8)


cache_file, ids_file = WORK / f"images_{IMG}.npy", WORK / f"ids_{IMG}.json"
ids = df["Example ID"].tolist()
if cache_file.exists() and ids_file.exists() and json.loads(ids_file.read_text()) == ids:
    X = np.load(cache_file)
    print("loaded cached images")
else:
    raw_ok = (PROJECT / df.path.iloc[0]).exists()
    if not raw_ok:
        raise FileNotFoundError(
            f"No image cache at {cache_file} and no raw images at {PROJECT / df.path.iloc[0]}.\n"
            "Upload either the raw data/raw/chest_xray folder or the dl_work/ cache folder (see the top of this notebook).")
    t0 = time.time()
    with ThreadPoolExecutor(8) as pool:
        X = np.stack(list(pool.map(load_gray, df.path)))
    np.save(cache_file, X); ids_file.write_text(json.dumps(ids))
    print(f"decoded {len(X)} images in {time.time() - t0:.0f}s")


def XY(ix):
    # images as (n, IMG, IMG, 1) uint8 and labels as float32; the model casts and scales internally
    return X[ix][..., None], y[ix]


print("images:", X.shape, X.dtype, "range", int(X.min()), "-", int(X.max()))
for s in ("train", "val", "test"):
    print(f"  {s:<5} n={len(idx[s]):>5}  pneumonia={y[idx[s]].mean():.1%}")

### Figure: class balance
**What it represents.** NORMAL vs PNEUMONIA counts in each split, and the pneumonia subtype mix.
**Why it matters.** Train/val are about 74% pneumonia but test is 62.5%, so a threshold tuned on validation sees a *different prevalence* at test time. The imbalance also motivates the class weights and the prevalence-independent threshold rule. (Uses the full index, so it looks the same in every profile.)

In [ ]:
order = ["train", "val", "test"]
cnt = index.groupby(["split", "label"]).size().unstack().loc[order]
sub = index.groupby(["split", "subtype"]).size().unstack().loc[order]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(3)
ax[0].bar(x - 0.2, cnt[0], 0.4, color=C["normal"], label="NORMAL")
ax[0].bar(x + 0.2, cnt[1], 0.4, color=C["pneu"], label="PNEUMONIA")
for i, s in enumerate(order):
    ax[0].text(i + 0.2, cnt.loc[s, 1], f"{cnt.loc[s, 1] / cnt.loc[s].sum():.1%}\npneumonia", ha="center", va="bottom", fontsize=9)
ax[0].set_xticks(x, order); ax[0].set_ylabel("images"); ax[0].set_title("Class balance per split"); ax[0].legend()
ax[0].set_ylim(0, cnt.values.max() * 1.18)
bottom = np.zeros(3)
for name, colour in (("normal", C["normal"]), ("bacterial", "#E8A87C"), ("viral", "#A0522D")):
    ax[1].bar(x, sub[name], 0.55, bottom=bottom, color=colour, label=name); bottom += sub[name].values
ax[1].set_xticks(x, order); ax[1].set_title("Composition incl. pneumonia subtype"); ax[1].legend()
plt.tight_layout(); savefig("01_class_balance")

## 2. Model building blocks

**Augmentation levels** (Keras layers, active only in training): `none`, `mild`, `strong`.

**Model** (Keras functional API):
`gray image -> 3 channels -> augmentation -> [rescale] -> pretrained backbone (global-average pooled) -> Dropout -> Dense(1, sigmoid)`.
- Last layer and loss follow the course's two-class recipe (Lab 5, Table 2): `Dense(1, activation="sigmoid")` with `binary_crossentropy`. A 2-unit softmax is mathematically equivalent; sigmoid gives one P(pneumonia), exactly what the predictions file and AUC need.
- Backbones (all `keras.applications`, ImageNet weights): **MobileNetV2** (small, ~2.3M parameters), **EfficientNetV2B0** (medium, ~5.9M), **ConvNeXtTiny** (large, ~27.8M). EfficientNetV2 and ConvNeXt rescale pixels internally; MobileNetV2 needs `Rescaling(1/127.5, offset=-1)`.
- `scratch` is a small 4-block CNN with no pretraining, kept **only as a labelled reference** (the course wants a TA's approval before training from scratch).

**Freezing** controls capacity: `unfreeze = 0` trains only the new head; `0.5` also trains the last half of the backbone layers; `1.0` trains all of it. **BatchNorm layers always stay frozen**, so their ImageNet statistics do not drift (standard Keras fine-tuning advice).

In [ ]:
def augmentation(level, hflip=False):
    if level == "none":
        return layers.Identity(name="augment")
    strength = dict(mild=(0.015, 0.05, 0.03, 0.10, 0.10), strong=(0.03, 0.15, 0.06, 0.25, 0.25))[level]
    rot, zoom, shift, bright, contrast = strength
    ls = [layers.RandomRotation(rot), layers.RandomZoom(zoom), layers.RandomTranslation(shift, shift),
          layers.RandomBrightness(bright, value_range=(0, 255)), layers.RandomContrast(contrast)]
    if hflip:
        ls.insert(0, layers.RandomFlip("horizontal"))
    return keras.Sequential(ls, name="augment")


BACKBONES = {"MobileNetV2": keras.applications.MobileNetV2,
             "EfficientNetV2B0": keras.applications.EfficientNetV2B0,
             "ConvNeXtTiny": keras.applications.ConvNeXtTiny}


def build_model(cfg):
    inp = keras.Input((IMG, IMG, 1), name="xray")
    # frozen 1x1 convolution with all-ones weights = copy the gray channel into 3 channels
    x = layers.Conv2D(3, 1, use_bias=False, kernel_initializer="ones", trainable=False, name="to_rgb")(inp)
    x = augmentation(cfg["aug"], cfg.get("hflip", False))(x)
    if cfg["backbone"] == "scratch":
        x = layers.Rescaling(1 / 255.0, name="pre")(x)
        for w in (16, 32, 64, 128):
            x = layers.Conv2D(w, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
            x = layers.MaxPooling2D()(x)
        x = layers.GlobalAveragePooling2D()(x)
    else:
        if cfg["backbone"] == "MobileNetV2":
            x = layers.Rescaling(1 / 127.5, offset=-1, name="pre")(x)
        base = BACKBONES[cfg["backbone"]](include_top=False, weights="imagenet" if cfg.get("pretrained", True) else None,
                                          input_shape=(IMG, IMG, 3), pooling="avg")
        x = base(x)
    x = layers.Dropout(cfg["dropout"], name="head_dropout")(x)
    out = layers.Dense(1, activation="sigmoid", dtype="float32", name="pneumonia")(x)
    return keras.Model(inp, out, name="xray_classifier")


def get_backbone(model):
    # the pretrained network is the nested Model layer; the augmentation block is also a (Sequential) Model, so skip it
    return next((l for l in model.layers if isinstance(l, keras.Model) and l.name != "augment"), None)


def set_trainable(model, frac, train_bn=False):
    # frac = share of the backbone's layers (counted from the END) that may be updated; 0 = head only.
    # BatchNorm layers stay frozen by default: with pretrained weights their stored statistics are already right.
    # A RANDOMLY initialised backbone needs train_bn=True (see also set_bn_momentum below): frozen BatchNorm keeps its
    # start-up statistics (mean 0, variance 1), the deep network then outputs a constant for every image and the
    # AUROC is exactly 0.5.
    base = get_backbone(model)
    if base is None:
        return
    base.trainable = frac > 0
    if frac > 0:
        cut = int(round(len(base.layers) * (1 - frac)))
        for i, l in enumerate(base.layers):
            l.trainable = (i >= cut) and (train_bn or not isinstance(l, layers.BatchNormalization))


def set_bn_momentum(model, momentum):
    # keras.applications backbones use BatchNorm momentum 0.999, i.e. moving averages that need ~1000 steps to settle.
    # When training from random weights for only a few hundred steps, the statistics used at inference time are still
    # close to their start-up values and the network outputs a constant (AUROC 0.5). A lower momentum tracks faster.
    base = get_backbone(model)
    for l in (base.layers if base is not None else []):
        if isinstance(l, layers.BatchNormalization):
            l.momentum = momentum


n_trainable = lambda m: int(sum(np.prod(w.shape) for w in m.trainable_weights))

# Overview: parameters, and how many are TRAINABLE at each unfreeze level (this also verifies that freezing works)
rows = []
for name in list(BACKBONES) + ["scratch"]:
    if name in P["backbones"] or name == "scratch":
        m = build_model(dict(backbone=name, pretrained=False, aug="mild", dropout=0.3))
        row = dict(backbone=name, parameters=m.count_params())
        for frac in (0.0, 0.5, 1.0):
            set_trainable(m, frac); row[f"trainable @ {frac:.0%}"] = n_trainable(m)
        if name != "scratch":
            assert row["trainable @ 0%"] < 3000 < row["trainable @ 50%"] < row["trainable @ 100%"], f"freezing does not work for {name}: {row}"
        rows.append(row)
display(pd.DataFrame(rows).style.format("{:,}", subset=[c for c in pd.DataFrame(rows).columns if c != "backbone"])); keras.backend.clear_session()

## 3. Metrics and sanity checks

The metric definitions match `eval/eval.py` (Positive = pneumonia). On top of its six metrics this notebook adds **MCC**, **AUROC**, **AUPRC**, **Brier score** and **expected calibration error**.

Sanity checks (Lab 5, "When something looks wrong"):
1. Look at the input the model actually receives: shape, value range, one image.
2. Labels agree with `eval/ground_truth.csv`; no patient is in both train and val.
3. **Majority-class baseline.** Always predicting pneumonia on the test set already gives F1 = 0.769, higher than the symbolic rule's 0.763, so F1 alone can not show a model learned anything. Specificity, balanced accuracy and MCC can.
4. The network must be able to **overfit a tiny batch**; if it can't, there is a bug.

In [ ]:
def binary_metrics(y_true, pred):
    y_true, pred = np.asarray(y_true).astype(int), np.asarray(pred).astype(int)
    tp = int(((y_true == 1) & (pred == 1)).sum()); fp = int(((y_true == 0) & (pred == 1)).sum())
    fn = int(((y_true == 1) & (pred == 0)).sum()); tn = int(((y_true == 0) & (pred == 0)).sum())
    div = lambda a, b: a / b if b else np.nan
    rec, spec, prec = div(tp, tp + fn), div(tn, tn + fp), div(tp, tp + fp)
    denom = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return {"F1": div(2 * tp, 2 * tp + fp + fn), "Recall": rec, "Specificity": spec, "Precision": prec,
            "Accuracy": div(tp + tn, len(y_true)), "BalAcc": (rec + spec) / 2,
            "MCC": (tp * tn - fp * fn) / denom if denom else 0.0}


def ece_score(y_true, p, bins=10):
    # expected calibration error: average |observed frequency - mean predicted probability| over probability bins
    ids_ = np.clip(np.digitize(p, np.linspace(0, 1, bins + 1)[1:-1]), 0, bins - 1)
    return sum(m.mean() * abs(y_true[m].mean() - p[m].mean()) for m in (ids_ == b for b in range(bins)) if m.any())


# 1. the input the model receives
xb, yb = XY(idx["train"][:8])
print(f"input batch: shape {xb.shape}, dtype {xb.dtype}, pixel range {xb.min()}-{xb.max()}; labels {yb.astype(int).tolist()}")

# 2. label / leakage checks
gt = pd.read_csv(PROJECT / "eval" / "ground_truth.csv").set_index("Example ID")["Ground Truth Label"]
test_df = df.iloc[idx["test"]].reset_index(drop=True)
assert (test_df["Example ID"].map(gt).eq("Positive").astype(int).values == test_df.label.values).all(), "labels disagree with ground_truth.csv"
assert not set(df.patient_id[idx["train"]]) & set(df.patient_id[idx["val"]]), "patient leakage train/val"
print("label check OK, no train/val patient overlap")
print("label counts  train:", dict(pd.Series(y[idx['train']]).value_counts().sort_index()),
      " val:", dict(pd.Series(y[idx['val']]).value_counts().sort_index()))

# 3. majority-class baseline (always predict pneumonia)
y_test = y[idx["test"]].astype(int)
print("\nAlways predict PNEUMONIA on the test set:", {k: round(v, 3) for k, v in binary_metrics(y_test, np.ones_like(y_test)).items()})

# 4. overfit a tiny batch
keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
small = idx["train"][:32]
xs, ys = XY(small)
tiny = build_model(dict(backbone="scratch", aug="none", dropout=0.0))
tiny.compile(keras.optimizers.Adam(3e-3), "binary_crossentropy", metrics=["accuracy"])
h = tiny.fit(xs, ys, epochs=60, batch_size=32, verbose=0)
print(f"\ntiny-batch overfit: accuracy {h.history['accuracy'][0]:.2f} -> {h.history['accuracy'][-1]:.2f}",
      "(OK)" if h.history["accuracy"][-1] >= 0.95 else "(WARNING: did not overfit; check the pipeline)")
del tiny; keras.backend.clear_session()

### Figure: sample images and augmentation
**What it represents.** Rows 1-2: random NORMAL and PNEUMONIA training images. Row 3: one pneumonia image and three random draws of the *strong* augmentation.
**Why it matters.** Shows what the network sees and that augmentation keeps the image plausible (still a chest, lungs still in frame). Pneumonia appears as hazy, whiter lung regions that are hard to separate from normal by eye, so the task is not trivial.

In [ ]:
rng = np.random.default_rng(SEED)
tr = idx["train"]
norm_ix = rng.choice(tr[y[tr] == 0], 4, replace=False)
pneu_ix = rng.choice(tr[y[tr] == 1], 4, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(11, 8.8))
for a, j in zip(axes[0], norm_ix):
    a.imshow(X[j], cmap="gray"); a.set_title(f"NORMAL  {df['Example ID'][j]}", color=C["normal"])
for a, j in zip(axes[1], pneu_ix):
    a.imshow(X[j], cmap="gray"); a.set_title(f"PNEUMONIA  {df['Example ID'][j]}", color=C["pneu"])
base_img = pneu_ix[0]
axes[2, 0].imshow(X[base_img], cmap="gray"); axes[2, 0].set_title("original")
aug_demo = keras.Sequential([layers.Conv2D(3, 1, use_bias=False, kernel_initializer="ones", trainable=False), augmentation("strong")])
inp = X[base_img][None, ..., None].astype("float32")
for a in axes[2, 1:]:
    a.imshow(np.clip(aug_demo(inp, training=True).numpy()[0, ..., 0], 0, 255), cmap="gray", vmin=0, vmax=255); a.set_title("strong augmentation")
for a in axes.ravel():
    a.axis("off")
plt.tight_layout(); savefig("02_samples_and_augmentation")

## 4. Training machinery

**One training run** (`train_run`) is the transfer-learning recipe:
1. **Phase A, head only.** Backbone frozen, only the new head trains (lr 1e-3, a few epochs). This stops a randomly initialised head from sending large gradients into, and wrecking, the pretrained features.
2. **Phase B, fine-tune.** Unfreeze the chosen share of the backbone and keep training with a small learning rate (about 1e-4) and **cosine decay**.
3. `unfreeze = 0` means "head only for the whole run"; `scratch` trains every layer from random weights.

Other ingredients:
- **Class weights** (`class_weight`) up-weight the rarer NORMAL class so the model can't win by always saying "pneumonia".
- **AdamW** optimiser (weight decay decoupled from the gradient step).
- **Keras metrics every epoch:** loss, accuracy, AUC and the confusion counts TP/FP/TN/FN, on train and validation. Recall, specificity, balanced accuracy and F1 per epoch are derived from the counts (at threshold 0.5). Training numbers are measured *during* the epoch (augmented, dropout on), so they look a bit pessimistic.
- The weights of the epoch with the best **validation AUROC** are kept and restored at the end, but training is **not** stopped early (unless `patience` is set), so the curves show what happens with longer training.
- Every run is **cached** (history, validation predictions, optionally the `.keras` model), so reruns and Colab reconnects are cheap.

In [ ]:
class BestTracker(keras.callbacks.Callback):
    # remembers the weights of the epoch with the best validation AUROC (keyed by variable path, so it
    # survives changes of which layers are trainable between phases)
    def __init__(self):
        super().__init__(); self.best, self.best_epoch, self.best_w, self.offset = -np.inf, 0, None, 0

    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get("val_auc")
        if v is not None and v > self.best:
            self.best, self.best_epoch = float(v), self.offset + epoch + 1
            self.best_w = {w.path: keras.ops.convert_to_numpy(w) for w in self.model.variables}
            assert len(self.best_w) == len(self.model.variables), "duplicate variable paths: weight restore would be wrong"

    def restore(self, model):
        if self.best_w is not None:
            for w in model.variables:
                w.assign(self.best_w[w.path])


class EpochLog(keras.callbacks.Callback):
    # records seconds per epoch and the learning rate used in each epoch
    def on_train_begin(self, logs=None):
        self.seconds, self.lrs = [], []

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()
        opt = getattr(self.model.optimizer, "inner_optimizer", self.model.optimizer)
        lr = opt.learning_rate
        self.lrs.append(float(keras.ops.convert_to_numpy(lr(opt.iterations) if callable(lr) else lr)))

    def on_epoch_end(self, epoch, logs=None):
        self.seconds.append(time.time() - self.t0)


def compile_model(model, lr, wd, total_steps):
    schedule = keras.optimizers.schedules.CosineDecay(lr, max(total_steps, 1), alpha=0.05)
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=schedule, weight_decay=wd), loss="binary_crossentropy",
                  metrics=[keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc"),
                           keras.metrics.TruePositives(name="tp"), keras.metrics.FalsePositives(name="fp"),
                           keras.metrics.TrueNegatives(name="tn"), keras.metrics.FalseNegatives(name="fn")])


def derive(h):
    # per-epoch recall, specificity, precision, balanced accuracy and F1 (at threshold 0.5) from the TP/FP/TN/FN counts
    d = h.copy()
    for p in ("", "val_"):
        tp, fp, tn, fn = (d[p + k] for k in ("tp", "fp", "tn", "fn"))
        d[p + "recall"] = tp / np.maximum(tp + fn, 1); d[p + "specificity"] = tn / np.maximum(tn + fp, 1)
        d[p + "precision"] = tp / np.maximum(tp + fp, 1); d[p + "balacc"] = (d[p + "recall"] + d[p + "specificity"]) / 2
        d[p + "f1"] = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    return d


def train_run(cfg, seed, train_ix=None, save_path=None, verbose=2):
    keras.backend.clear_session(); keras.utils.set_random_seed(seed)
    tr = idx["train"] if train_ix is None else train_ix
    (Xt, yt), (Xv, yv) = XY(tr), XY(idx["val"])
    n0, n1 = int((yt == 0).sum()), int((yt == 1).sum())
    cw = {0: len(yt) / (2 * n0), 1: len(yt) / (2 * n1)} if cfg.get("class_weight", True) else None
    steps = int(np.ceil(len(tr) / BATCH))
    model = build_model(cfg)
    tracker, parts, t0 = BestTracker(), [], time.time()

    def run_phase(phase, epochs, lr):
        if epochs <= 0:
            return
        compile_model(model, lr, cfg["wd"], epochs * steps)
        log = EpochLog()
        cbs = [tracker, log]
        if cfg.get("patience"):
            cbs.append(keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=cfg["patience"]))
        # Keras applies class_weight to the TRAINING loss only; give the validation set the same per-sample weights
        # so that train loss and validation loss are measured on the same scale (metrics such as AUROC stay unweighted)
        val_data = (Xv, yv) if cw is None else (Xv, yv, np.where(yv == 1, cw[1], cw[0]).astype('float32'))
        hist = model.fit(Xt, yt, validation_data=val_data, epochs=epochs, batch_size=BATCH, class_weight=cw,
                         callbacks=cbs, verbose=verbose)
        d = pd.DataFrame(hist.history); d["phase"] = phase
        d["lr"], d["seconds"] = log.lrs[:len(d)], log.seconds[:len(d)]
        tracker.offset += len(d); parts.append(d)

    if cfg["backbone"] == "scratch":
        run_phase("train", cfg["epochs"], cfg["lr"])
    elif not cfg.get("pretrained", True):
        set_trainable(model, 1.0, train_bn=cfg.get("train_bn", False))
        if cfg.get("bn_momentum") is not None:
            set_bn_momentum(model, cfg["bn_momentum"])
        run_phase("train", cfg["epochs"], cfg["lr"])
    elif cfg["unfreeze"] == 0:
        set_trainable(model, 0.0); run_phase("head", cfg["head_epochs"] + cfg["epochs"], 1e-3)
    else:
        set_trainable(model, 0.0); run_phase("head", cfg["head_epochs"], 1e-3)
        set_trainable(model, cfg["unfreeze"]); run_phase("finetune", cfg["epochs"], cfg["lr"])
    tracker.restore(model)
    hist_df = pd.concat(parts, ignore_index=True); hist_df.insert(0, "epoch", np.arange(1, len(hist_df) + 1))
    val_prob = model.predict(Xv, batch_size=64, verbose=0).ravel()
    out = dict(history=hist_df, val_prob=val_prob, best_auc=float(tracker.best), best_epoch=int(tracker.best_epoch),
               seconds=time.time() - t0, params=int(model.count_params()), trainable_params=int(n_trainable(model)), n_train=len(tr))
    if save_path is not None:
        model.compile(optimizer=keras.optimizers.Adam(), loss="binary_crossentropy")   # fresh optimizer: keeps the file small
        model.save(save_path)
    return out


ALL_RUNS, STAGE = [], "setup"


def run_id(cfg, seed, n):
    h = hashlib.md5(json.dumps([cfg, seed, n, IMG, MODE, P["batch"]], sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{cfg['backbone']}_u{cfg.get('unfreeze', 1)}_{h}"


def cached_run(cfg, seed=SEED, train_ix=None, save_model=False, verbose=2):
    n = None if train_ix is None else int(len(train_ix))
    rid = run_id(cfg, seed, n); d = RUN_DIR / rid; mp = d / "model.keras"
    if (d / "meta.json").exists() and not RERUN and (not save_model or mp.exists()):
        res = json.loads((d / "meta.json").read_text())
        res["history"], res["val_prob"] = pd.read_csv(d / "history.csv"), np.load(d / "val_prob.npy")
    else:
        d.mkdir(parents=True, exist_ok=True)
        print(f"--> training {rid}  ({json.dumps({k: v for k, v in cfg.items() if k != 'patience'}, default=str)}, seed {seed})")
        res = train_run(cfg, seed, train_ix, str(mp) if save_model else None, verbose)
        res["history"].to_csv(d / "history.csv", index=False); np.save(d / "val_prob.npy", res["val_prob"])
        (d / "meta.json").write_text(json.dumps({k: v for k, v in res.items() if k not in ("history", "val_prob")}))
    res["cfg"], res["seed"], res["id"], res["model_path"] = cfg, seed, rid, (str(mp) if mp.exists() else None)
    ALL_RUNS.append(dict(stage=STAGE, id=rid, seconds=res["seconds"], epochs=len(res["history"])))
    return res


def pick_simplest(frame, score="val_auroc", complexity="complexity", tol=P["tol"]):
    # best score wins, but any run within `tol` of the best counts as tied and the SIMPLER one is chosen
    ok = frame[frame[score] >= frame[score].max() - tol]
    return ok.sort_values([complexity, score], ascending=[True, False]).index[0]


DEFAULT = dict(backbone="EfficientNetV2B0", pretrained=True, unfreeze=0.5, lr=1e-4, wd=1e-4, dropout=0.3, aug="mild",
               hflip=False, class_weight=True, head_epochs=P["head_epochs"], epochs=P["ep_sweep"], patience=None)
print("default configuration:", DEFAULT)

## 5. Tuning (validation only)

Tuning is **staged**, not a full grid (a grid would be hundreds of runs). Each stage settles one decision, then the next stage starts from that winner.

**How a winner is chosen.** Best validation AUROC wins, but validation has only about 1,000 images, so differences under about 0.01 AUROC are noise. **Within that tolerance the simpler model wins** (fewer trainable layers or parameters): it is cheaper to run and less likely to overfit, as the Lab 5 handout advises.
The test set is never used here.

### Stage 1: capacity (how much of the backbone is unfrozen)
This is the capacity hyperparameter the course asks for. `0%` = head only (least capacity, may underfit); `100%` = every layer except BatchNorm (most capacity, may overfit). The sweep trains for the same number of epochs with **no early stopping**, so overfitting becomes visible in the curves.

In [ ]:
STAGE = "1 capacity sweep"
s1_runs, s1_rows = {}, []
for frac in P["fracs"]:
    r = cached_run({**DEFAULT, "unfreeze": frac})
    s1_runs[frac] = r
    s1_rows.append(dict(unfreeze=frac, trainable_params=r["trainable_params"], val_auroc=r["best_auc"], best_epoch=r["best_epoch"],
                        minutes=r["seconds"] / 60, complexity=frac))
s1 = pd.DataFrame(s1_rows)
scratch_run = None
if P["include_scratch"]:       # REFERENCE ONLY: not eligible to be selected
    scratch_run = cached_run({**DEFAULT, "backbone": "scratch", "pretrained": False, "unfreeze": 1.0, "lr": 1e-3,
                              "head_epochs": 0, "epochs": P["ep_sweep"] + P["head_epochs"]})
best_frac = float(s1.loc[pick_simplest(s1), "unfreeze"])
display(s1.drop(columns="complexity").style.format({"val_auroc": "{:.4f}", "minutes": "{:.1f}", "trainable_params": "{:,}"}))
if scratch_run:
    print(f"scratch CNN reference: val AUROC {scratch_run['best_auc']:.4f} (best epoch {scratch_run['best_epoch']})")
print(f"-> stage 1 winner: unfreeze {best_frac:.0%} of the backbone")
s1.drop(columns="complexity").to_csv(RES_DIR / "stage1_capacity.csv", index=False)

### Figure: capacity vs overfitting
**What it represents.** One curve per unfreezing level (and the scratch CNN as a dashed reference): validation loss, validation AUROC, and the **train minus validation AUROC gap** at every epoch.
**How to read it.** (Early in training the gap can be *negative*: training metrics are measured while augmentation and dropout are on, validation metrics are not.) More capacity usually means the training score pulls further ahead of validation (bigger gap, rising validation loss = overfitting). Head-only models stay close but may plateau lower (underfitting). The best choice is where validation AUROC is highest without a runaway gap. This is the bias-variance picture the course asks you to tune.

In [ ]:
def plot_capacity(runs, labels, scratch=None, fname="03_capacity_vs_overfitting"):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    cols = plt.cm.viridis(np.linspace(0.05, 0.85, len(runs)))
    for (r, lab), c in zip(zip(runs, labels), cols):
        h = r["history"]
        axs[0].plot(h.epoch, h.val_loss, c=c, label=lab); axs[1].plot(h.epoch, h.val_auc, c=c, label=lab)
        axs[2].plot(h.epoch, h.auc - h.val_auc, c=c, label=lab)
    if scratch is not None:
        h = scratch["history"]
        for a, v in zip(axs, [h.val_loss, h.val_auc, h.auc - h.val_auc]):
            a.plot(h.epoch, v, c="grey", ls="--", label="scratch CNN (reference)")
    for a, t in zip(axs, ["validation loss (lower is better)", "validation AUROC", "train AUROC minus validation AUROC (overfitting gap)"]):
        a.set_title(t, fontsize=10); a.set_xlabel("epoch"); a.xaxis.set_major_locator(MaxNLocator(integer=True))
    axs[2].axhline(0, c="k", lw=0.5); axs[1].legend(fontsize=8, loc="lower right")
    plt.tight_layout(); savefig(fname)


plot_capacity([s1_runs[f] for f in P["fracs"]], [f"{f:.0%} unfrozen" for f in P["fracs"]], scratch_run)

### Stage 2: backbone size
Same recipe and the stage-1 winner for unfreezing, but a different pretrained model: small (MobileNetV2), medium (EfficientNetV2B0), large (ConvNeXtTiny). Ties again go to the smaller model. It is common practice to compare a few candidate backbones; on about 4,000 medical images the differences are often within validation noise, and what often decides is **cost** (see the cost figure later).

In [ ]:
STAGE = "2 backbone comparison"
s2_runs, s2_rows = {}, []
for bb in P["backbones"]:
    r = cached_run({**DEFAULT, "backbone": bb, "unfreeze": best_frac})
    s2_runs[bb] = r
    s2_rows.append(dict(backbone=bb, params=r["params"], val_auroc=r["best_auc"], best_epoch=r["best_epoch"],
                        sec_per_epoch=r["history"].seconds.mean(), complexity=r["params"]))
s2 = pd.DataFrame(s2_rows)
best_backbone = s2.loc[pick_simplest(s2), "backbone"]
display(s2.drop(columns="complexity").style.format({"val_auroc": "{:.4f}", "params": "{:,}", "sec_per_epoch": "{:.1f}"}))
print(f"-> stage 2 winner: {best_backbone}")
s2.drop(columns="complexity").to_csv(RES_DIR / "stage2_backbones.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for bb, c in zip(P["backbones"], plt.cm.Set2(np.arange(len(P["backbones"])))):
    h = s2_runs[bb]["history"]
    axs[0].plot(h.epoch, h.val_auc, c=c, label=f"{bb} ({s2_runs[bb]['params'] / 1e6:.1f}M params)")
    axs[1].plot(h.epoch, h.val_loss, c=c, label=bb)
axs[0].set_title("Validation AUROC per epoch"); axs[1].set_title("Validation loss per epoch")
for a in axs: a.set_xlabel("epoch"); a.xaxis.set_major_locator(MaxNLocator(integer=True))
axs[0].legend(fontsize=8)
plt.tight_layout(); savefig("04_backbone_comparison")

### Stage 3: small settings (random search)
With backbone and unfreezing fixed, search the remaining knobs: learning rate, weight decay, head dropout and augmentation strength. **Random search** beats a grid for the same budget because every trial tries a new value of *every* knob. The defaults are re-trained for the same (shorter) budget as a fair baseline.

In [ ]:
STAGE = "3 random search"
base3 = {**DEFAULT, "backbone": best_backbone, "unfreeze": best_frac, "epochs": P["ep_tune"]}
rng3 = random.Random(SEED)
cfgs3 = [("defaults", base3)] + [(f"trial {i + 1}", {**base3, "lr": float(10 ** rng3.uniform(-5, -3.3)), "wd": float(10 ** rng3.uniform(-6, -2)),
                                                 "dropout": rng3.choice([0.0, 0.2, 0.4, 0.5]), "aug": rng3.choice(["none", "mild", "strong"])})
                                 for i in range(P["n_trials"])]
s3_rows, s3_cfg = [], {}
for name, cfg in cfgs3:
    r = cached_run(cfg); s3_cfg[name] = cfg
    s3_rows.append(dict(trial=name, lr=cfg["lr"], wd=cfg["wd"], dropout=cfg["dropout"], aug=cfg["aug"],
                        val_auroc=r["best_auc"], best_epoch=r["best_epoch"], minutes=r["seconds"] / 60))
s3 = pd.DataFrame(s3_rows).sort_values("val_auroc", ascending=False).reset_index(drop=True)
best_trial = s3.loc[0, "trial"]
best_cfg = s3_cfg[best_trial]
display(s3.style.format({"lr": "{:.1e}", "wd": "{:.1e}", "val_auroc": "{:.4f}", "minutes": "{:.1f}"}))
print(f"-> stage 3 winner: {best_trial}: lr={best_cfg['lr']:.1e} wd={best_cfg['wd']:.1e} dropout={best_cfg['dropout']} aug={best_cfg['aug']}")
s3.to_csv(RES_DIR / "stage3_random_search.csv", index=False)

### Figure: hyper-parameter search
**What it represents.** Validation AUROC of each trial against learning rate, weight decay and dropout (colour = augmentation strength).
**How to read it.** A visible trend (for example a good learning-rate band) means that knob matters; a flat cloud means the model is not sensitive to it. It documents that tuning was systematic.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
aug_col = {"none": "#7f7f7f", "mild": "#4C72B0", "strong": "#DD8452"}
for ax, col, logx in zip(axes, ["lr", "wd", "dropout"], [True, True, False]):
    for _, r in s3.iterrows():
        ax.scatter(r[col], r.val_auroc, c=aug_col[r.aug], s=70, edgecolor="k", linewidth=0.5)
        if r.trial == "defaults": ax.annotate("defaults", (r[col], r.val_auroc), textcoords="offset points", xytext=(5, 5), fontsize=8)
    if logx: ax.set_xscale("log")
    ax.set_xlabel(col)
axes[0].set_ylabel("validation AUROC")
fig.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=c, label=f"aug: {a}") for a, c in aug_col.items()],
           loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
fig.suptitle("Random search: validation AUROC per trial", y=1.02)
plt.tight_layout(); savefig("05_hyperparameter_search")

## 6. Longer training (Stage 4)

The winning configuration is trained for **many more epochs, with no early stopping**, and repeated with several random seeds. Seeds give a feel for run-to-run noise; the mean and spread over seeds are plotted.
The weights of the best-validation-AUROC epoch are kept as the final model (saved with `model.save`); the later epochs are there to show what longer training does.

In [ ]:
STAGE = "4 long training"
long_cfg = {**best_cfg, "epochs": P["ep_long"], "patience": None}
long_runs = [cached_run(long_cfg, seed=SEED + i, save_model=True) for i in range(P["long_seeds"])]
lt = pd.DataFrame([dict(seed=r["seed"], best_val_auroc=r["best_auc"], best_epoch=r["best_epoch"], total_epochs=len(r["history"]),
                        minutes=r["seconds"] / 60) for r in long_runs])
display(lt.style.format({"best_val_auroc": "{:.4f}", "minutes": "{:.1f}"}))
final = max(long_runs, key=lambda r: r["best_auc"])
FINAL_PATH = MODEL_DIR / "deep_learning_final.keras"
shutil.copy(final["model_path"], FINAL_PATH)
(RES_DIR / "best_config.json").write_text(json.dumps(long_cfg, indent=2, default=str))
lt.to_csv(RES_DIR / "long_training_runs.csv", index=False)
print(f"final model: seed {final['seed']}, best epoch {final['best_epoch']} of {len(final['history'])}, val AUROC {final['best_auc']:.4f}")
print("saved with model.save ->", FINAL_PATH.relative_to(PROJECT) if FINAL_PATH.is_relative_to(PROJECT) else FINAL_PATH)

### Figure: training curves over long training
**What it represents.** Per epoch (mean and band over seeds): loss, AUROC, validation recall and specificity, balanced accuracy and F1, precision, and the learning-rate schedule. Dashed = training, solid = validation. The dotted vertical line is the end of head-only training; the star marks the epoch whose weights are kept.
**How to read it.** This is the course's "training curve". Training improving while validation stalls or gets worse means overfitting; both low means underfitting. The recall/specificity panel shows *what kind* of mistakes change as training goes on. (Metrics here are at threshold 0.5, so they can differ from the tuned-threshold numbers later.)

In [ ]:
def stack(runs, col):
    hs = [derive(r["history"]) for r in runs]
    arr = np.stack([h[col].values for h in hs]); return hs[0].epoch.values, arr.mean(0), arr.std(0)


def band(ax, runs, col, colour, label=None, ls="-"):
    ep, m, s = stack(runs, col)
    ax.plot(ep, m, c=colour, ls=ls, label=label); ax.fill_between(ep, m - s, m + s, color=colour, alpha=0.15)


fig, axs = plt.subplots(2, 3, figsize=(15, 8))
band(axs[0, 0], long_runs, "loss", C["dl"], "train", "--"); band(axs[0, 0], long_runs, "val_loss", C["dl"], "validation"); axs[0, 0].set_title("Loss (class-weighted BCE)")
band(axs[0, 1], long_runs, "auc", C["dl"], "train", "--"); band(axs[0, 1], long_runs, "val_auc", C["dl"], "validation"); axs[0, 1].set_title("AUROC")
band(axs[0, 2], long_runs, "val_recall", C["pneu"], "recall (pneumonia caught)"); band(axs[0, 2], long_runs, "val_specificity", C["normal"], "specificity (normal cleared)")
axs[0, 2].set_title("Validation recall and specificity @0.5")
band(axs[1, 0], long_runs, "val_balacc", C["dl"], "balanced accuracy"); band(axs[1, 0], long_runs, "val_f1", C["scratch"], "F1"); axs[1, 0].set_title("Validation balanced accuracy and F1 @0.5")
band(axs[1, 1], long_runs, "val_precision", C["symbolic"], "precision"); band(axs[1, 1], long_runs, "val_acc", C["human"], "accuracy"); axs[1, 1].set_title("Validation precision and accuracy @0.5")
ep, lr_m, _ = stack(long_runs, "lr"); axs[1, 2].plot(ep, lr_m, c="k"); axs[1, 2].set_yscale("log"); axs[1, 2].set_title("Learning rate (head phase, then cosine decay)")
ph = long_runs[0]["history"]; n_head = int((ph.phase == "head").sum())
for a in axs.ravel():
    a.set_xlabel("epoch"); a.xaxis.set_major_locator(MaxNLocator(integer=True))
    if n_head and n_head < len(ph): a.axvline(n_head + 0.5, c="grey", ls=":")
    if a is not axs[1, 2]: a.legend(fontsize=8)
for r in long_runs:
    axs[0, 1].scatter([r["best_epoch"]], [r["best_auc"]], marker="*", s=200, c="gold", edgecolor="k", zorder=5)
fig.suptitle(f"Long training: {len(long_runs)} seeds x {len(ph)} epochs, {best_backbone}", y=1.0)
plt.tight_layout(); savefig("06_long_training_curves")

### Figure: training loss vs validation loss
**What it represents.** Exactly two lines, averaged over the seeds of the long run: loss on the training set (dashed) and loss on the validation set (solid), both class-weighted binary cross-entropy, one point per epoch.
**How to read it.** Both falling together = the model is learning. Training loss keeps falling while validation loss flattens or rises = overfitting. (Training loss is measured while the network is training, with augmentation and dropout on, so it can sit slightly above validation loss early on.)

In [ ]:
ep, tl, _ = stack(long_runs, "loss"); _, vl, _ = stack(long_runs, "val_loss")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ep, tl, c=C["dl"], ls="--", label="training loss"); ax.plot(ep, vl, c=C["dl"], label="validation loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss (class-weighted binary cross-entropy)"); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title("Training vs validation loss"); ax.legend()
plt.tight_layout(); savefig("06b_train_vs_validation_loss")

### Figure: generalization gap and the "sweet spot"
**What it represents.** Left: how far training AUROC is ahead of validation AUROC at each epoch. Right: validation **loss** (blue) against validation **AUROC** (red) on twin axes, with the epoch of minimum loss and of maximum AUROC marked.
**How to read it.** A growing gap means the model is memorising the training set (it can start negative because training metrics are measured with augmentation and dropout on). A well-known effect: validation *loss* often starts rising (the probabilities become over-confident) while AUROC (ranking quality) stays flat or still improves, so "when to stop" depends on what you measure. The gap between the two marked epochs is the practical answer to "does longer training help?".

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.4))
ep = derive(long_runs[0]["history"]).epoch.values
gap = np.stack([(derive(r["history"]).auc - derive(r["history"]).val_auc).values for r in long_runs])
axs[0].plot(ep, gap.mean(0), c=C["dl"]); axs[0].fill_between(ep, gap.mean(0) - gap.std(0), gap.mean(0) + gap.std(0), color=C["dl"], alpha=0.15)
axs[0].axhline(0, c="k", lw=0.5); axs[0].set_title("Train AUROC minus validation AUROC"); axs[0].set_xlabel("epoch"); axs[0].set_ylabel("gap")
ax1 = axs[1]; ax2 = ax1.twinx(); ax2.grid(False); ax2.spines["right"].set_visible(True)
_, vl, vls = stack(long_runs, "val_loss"); _, va, vas = stack(long_runs, "val_auc")
ax1.plot(ep, vl, c=C["normal"]); ax1.fill_between(ep, vl - vls, vl + vls, color=C["normal"], alpha=0.15); ax1.set_ylabel("validation loss", color=C["normal"])
ax2.plot(ep, va, c=C["dl"]); ax2.fill_between(ep, va - vas, va + vas, color=C["dl"], alpha=0.15); ax2.set_ylabel("validation AUROC", color=C["dl"])
e_loss, e_auc = int(ep[np.argmin(vl)]), int(ep[np.argmax(va)])
ax1.axvline(e_loss, c=C["normal"], ls="--"); ax2.axvline(e_auc, c=C["dl"], ls="--")
ax1.set_title(f"Min val loss at epoch {e_loss}; max val AUROC at epoch {e_auc}"); ax1.set_xlabel("epoch")
plt.tight_layout(); savefig("07_generalization_gap")

### Figure: what each extra epoch buys
**What it represents.** Validation AUROC, loss and balanced accuracy at fixed checkpoints during the long run (mean over seeds, error bars = std), next to the value at the best epoch.
**How to read it.** If the bars flatten (or fall) after some epoch, training longer no longer helps. Choosing the checkpoint on *validation* data is exactly what the best-epoch rule does; this figure shows how much that choice matters.

In [ ]:
total = len(long_runs[0]["history"])
snap = [e for e in (5, 10, 20, 30, 40, 50, 60) if e < total] + [total]
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title, better in zip(axs, ["val_auc", "val_loss", "val_balacc"], ["Validation AUROC", "Validation loss", "Validation balanced accuracy @0.5"], [max, min, max]):
    vals = np.array([[derive(r["history"])[col].values[e - 1] for e in snap] for r in long_runs])
    ax.bar([str(e) for e in snap], vals.mean(0), yerr=vals.std(0), color=C["dl"], alpha=0.8, capsize=3)
    best_vals = [better(derive(r["history"])[col]) for r in long_runs]
    ax.axhline(np.mean(best_vals), c="k", ls="--", label=f"best epoch (mean {np.mean(best_vals):.3f})")
    ax.set_ylim(max(0, min(vals.min(), np.min(best_vals)) - 0.05), min(1.05 if col != "val_loss" else 10, max(vals.max(), np.max(best_vals)) + 0.05))
    ax.set_title(title); ax.set_xlabel("epochs trained"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("08_epoch_snapshots")

## 7. Decision threshold (**your decision**)

The model outputs a probability; a **threshold** turns it into Positive/Negative. 0.5 is not neutral here: the loss is class-weighted and the test set has a different prevalence than validation. The threshold is chosen on **validation data only** and then frozen.

`choose_threshold` uses **Youden's J = recall + specificity - 1**. It does not depend on prevalence, so it survives the shift from about 74% pneumonia (val) to 62.5% (test). Alternatives and their trade-offs:

| Rule | Behaviour |
|---|---|
| maximise **F1** on val | favours recall at 74% prevalence; likely more false positives on test |
| maximise **balanced accuracy** | essentially the same as Youden's J |
| **recall >= 0.95**, then best specificity | clinically motivated (a missed pneumonia is the costly error) but lower specificity |

Edit the function if you prefer another rule and state the choice in the write-up.

In [ ]:
def choose_threshold(y_val, p_val):
    # YOUR DECISION: replace this body to use another rule (see the table above).
    fpr, tpr, thr = roc_curve(y_val, p_val)
    return float(thr[np.argmax(tpr - fpr)])      # Youden's J


yv = y[idx["val"]].astype(int)
p_val = final["val_prob"]
THR = choose_threshold(yv, p_val)
m = binary_metrics(yv, p_val >= THR)
print(f"threshold {THR:.3f}  ->  validation recall {m['Recall']:.3f}, specificity {m['Specificity']:.3f}, balanced acc {m['BalAcc']:.3f}, F1 {m['F1']:.3f}")

## 8. Run the test set ONCE

Everything above used only train and validation. This cell loads the saved final model and predicts the test set **one time**; the probabilities are written to `results/test_scores.csv`. If that file already exists the cell **refuses to run again** and just reloads it (set `ALLOW_SECOND_TEST_RUN = True` at the top only if you really intend to start over; doing it to chase a better score defeats the purpose of a test set).

The predictions file `eval/predictions/deep_learning_eval.csv` has the columns the other approaches use (`Example ID`, `Deep Learning Prediction`) plus the probability, so `eval/eval.py` can score it next to the human and symbolic approaches. The model only ever sees the pixels, the same input as the human and the symbolic rule.

In [ ]:
TEST_CSV = RES_DIR / "test_scores.csv"
if TEST_CSV.exists() and not ALLOW_SECOND_TEST_RUN:
    print("The test set was already evaluated; loading the saved scores (no new test-set run).")
    scores = pd.read_csv(TEST_CSV)
else:
    final_model = keras.saving.load_model(FINAL_PATH, compile=False)
    Xte, _ = XY(idx["test"])
    t0 = time.time()
    p_test = final_model.predict(Xte, batch_size=64, verbose=0).ravel()
    print(f"predicted {len(Xte)} test images in {time.time() - t0:.1f}s on {DEVICE}")
    scores = pd.DataFrame({"Example ID": test_df["Example ID"], "label": y_test, "subtype": test_df.subtype, "p_deep": p_test})
    scores.to_csv(TEST_CSV, index=False)
assert (scores["Example ID"].values == test_df["Example ID"].values).all()
p_test = scores.p_deep.values
pred_dl = (p_test >= THR).astype(int)

if MODE == "full":
    pd.DataFrame({"Example ID": scores["Example ID"], "Deep Learning Prediction": np.where(pred_dl == 1, "Positive", "Negative"),
                  "Deep Learning Probability": np.round(p_test, 6)}).sort_values("Example ID").to_csv(
        PROJECT / "eval" / "predictions" / "deep_learning_eval.csv", index=False)
    print("wrote eval/predictions/deep_learning_eval.csv")
else:
    print("quick profile: eval/predictions/deep_learning_eval.csv NOT written")

### Metrics with confidence intervals
Compared with: *always positive* (the trivial baseline), the **symbolic rule** (`eval/predictions/symbolic_eval.csv`) and, on the 40-image human subset, the **human baseline**.
**95% confidence intervals** come from a bootstrap that resamples *patients*, not images: several X-rays can belong to one child, so resampling single images would make the intervals too narrow. NORMAL images have no patient ID, so each counts as its own group.

In [ ]:
sym = pd.read_csv(PROJECT / "eval" / "predictions" / "symbolic_eval.csv").set_index("Example ID")["Algorithmic Prediction"]
pred = {"Always positive": np.ones(len(y_test), int),
        "Symbolic rule": (test_df["Example ID"].map(sym) == "Positive").astype(int).values,
        "Deep learning": pred_dl}
probs = {"Deep learning": p_test}
APPS = list(pred)
members = list(pd.DataFrame({"g": test_df.patient_id.values}).groupby("g").indices.values())


def stats(ix, name):
    d = binary_metrics(y_test[ix], pred[name][ix])
    if name in probs and len(np.unique(y_test[ix])) == 2:
        d["AUROC"] = roc_auc_score(y_test[ix], probs[name][ix]); d["AUPRC"] = average_precision_score(y_test[ix], probs[name][ix])
    return d


def bootstrap(name):
    rng = np.random.default_rng(SEED); out = []
    for _ in range(P["n_boot"]):
        pick = rng.integers(0, len(members), len(members))
        out.append(stats(np.concatenate([members[i] for i in pick]), name))
    b = pd.DataFrame(out)
    return b.quantile(0.025), b.quantile(0.975)


point, lo, hi = {}, {}, {}
for a in APPS:
    point[a] = stats(np.arange(len(y_test)), a)
    if a in probs:
        point[a].update(Brier=brier_score_loss(y_test, probs[a]), ECE=ece_score(y_test.astype(float), probs[a]))
    lo[a], hi[a] = bootstrap(a)
cols = ["F1", "Recall", "Specificity", "Precision", "Accuracy", "BalAcc", "MCC", "AUROC", "AUPRC", "Brier", "ECE"]
results = pd.DataFrame(point).T.reindex(columns=cols)
ci_lo, ci_hi = pd.DataFrame(lo).T, pd.DataFrame(hi).T
fmt = results.copy().astype(object)
for a in APPS:
    for c in cols:
        v = results.loc[a, c]
        if pd.isna(v): fmt.loc[a, c] = "-"
        elif c in ci_lo.columns and not pd.isna(ci_lo.loc[a, c]): fmt.loc[a, c] = f"{v:.3f} [{ci_lo.loc[a, c]:.3f}, {ci_hi.loc[a, c]:.3f}]"
        else: fmt.loc[a, c] = f"{v:.3f}"
print(f"Test set (n = {len(y_test)}), threshold {THR:.3f}, 95% patient-level bootstrap CI in brackets ({P['n_boot']} resamples)")
display(fmt[["F1", "Recall", "Specificity", "BalAcc", "MCC", "AUROC", "AUPRC"]]); display(fmt[["Precision", "Accuracy", "Brier", "ECE"]])
csv_out = results.copy()
for c in ci_lo.columns: csv_out[f"{c}_ci_lo"], csv_out[f"{c}_ci_hi"] = ci_lo[c], ci_hi[c]
csv_out.to_csv(RES_DIR / "test_metrics.csv")

# ---- the 40-image human-baseline subset
subset_ids = set(pd.read_csv(PROJECT / "approaches" / "1_human_baseline" / "subset_ids.csv")["Example ID"])
human = pd.read_csv(PROJECT / "eval" / "predictions" / "human_eval.csv").set_index("Example ID")["Human Prediction"]
mask = test_df["Example ID"].isin(subset_ids).values
subset_res = {}
if mask.sum() >= 10:
    subset_res["Human"] = binary_metrics(y_test[mask], (test_df.loc[mask, "Example ID"].map(human) == "Positive").astype(int).values)
    for a in APPS[1:]: subset_res[a] = binary_metrics(y_test[mask], pred[a][mask])
    print(f"\nHuman-baseline subset (n = {mask.sum()})")
    display(pd.DataFrame(subset_res).T[["F1", "Recall", "Specificity", "Precision", "Accuracy", "BalAcc", "MCC"]].round(3))
    pd.DataFrame(subset_res).T.to_csv(RES_DIR / "human_subset_metrics.csv")
else:
    print(f"\nonly {mask.sum()} human-subset images in this run: subset comparison skipped")

## 9. Figures: task performance and failure modes

### Confusion matrices
**What they represent.** Counts and row-normalised rates of true/false positives and negatives on the test set (the deep model at its validation-chosen threshold).
**How to read them.** The bottom-left cell is the **missed pneumonia** rate, the clinically costly error; the top-right is the false-alarm rate. Row percentages are specificity (top row) and recall (bottom row), so models stay comparable despite imbalanced classes.

In [ ]:
def plot_cm(ax, y_true, p, title):
    cm = np.array([[((y_true == a) & (p == b)).sum() for b in (0, 1)] for a in (0, 1)])
    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n{pct[i, j]:.0%}", ha="center", va="center", fontsize=13, color="white" if pct[i, j] > 0.5 else "black")
    ax.set_xticks([0, 1], ["Normal", "Pneumonia"]); ax.set_yticks([0, 1], ["Normal", "Pneumonia"])
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title(title); ax.grid(False)


fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, a in zip(axes, ["Symbolic rule", "Deep learning"]): plot_cm(ax, y_test, pred[a], a)
plt.tight_layout(); savefig("09_confusion_matrices")

### ROC curve
**What it represents.** True-positive rate (recall) against false-positive rate (1 - specificity) as the threshold sweeps from 1 to 0. The area under it (AUROC) is the probability that a random pneumonia image scores higher than a random normal one.
**How to read it.** Higher and further left is better; the diagonal is guessing. The filled circle is the deep model's operating point at its chosen threshold. The symbolic rule and the human only give hard yes/no answers, so they appear as single points (human: 40-image subset only). A curve above those points ranks cases better at *every* operating point. AUC is a secondary metric because it can only be computed for the deep model.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6))
fpr, tpr, _ = roc_curve(y_test, p_test)
ax.plot(fpr, tpr, c=C["dl"], lw=2, label=f"Deep learning (AUROC {results.loc['Deep learning', 'AUROC']:.3f})")
ax.scatter(1 - results.loc["Deep learning", "Specificity"], results.loc["Deep learning", "Recall"], c=C["dl"], s=70, edgecolor="k", zorder=5, label="deep learning @ chosen threshold")
ax.scatter(1 - results.loc["Symbolic rule", "Specificity"], results.loc["Symbolic rule", "Recall"], marker="D", s=80, c=C["symbolic"], edgecolor="k", label="Symbolic rule", zorder=5)
if "Human" in subset_res:
    ax.scatter(1 - subset_res["Human"]["Specificity"], subset_res["Human"]["Recall"], marker="s", s=80, c="white", edgecolor="k", label=f"Human ({mask.sum()}-image subset)", zorder=5)
ax.plot([0, 1], [0, 1], "k:", lw=1, label="chance")
ax.set_xlabel("false-positive rate (1 - specificity)"); ax.set_ylabel("true-positive rate (recall)")
ax.set_title("ROC curve, test set"); ax.legend(loc="lower right", fontsize=8); ax.set_aspect("equal")
plt.tight_layout(); savefig("10_roc_curve")

### Precision-recall curve
**What it represents.** Precision (how many flagged cases really are pneumonia) against recall as the threshold changes; AUPRC summarises it.
**How to read it.** The dotted line is the precision of a useless classifier (the share of pneumonia in the test set). The further above it the curve stays, the better. PR curves are more informative than ROC when classes are imbalanced because they ignore true negatives.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
pr, rc, _ = precision_recall_curve(y_test, p_test)
ax.plot(rc, pr, c=C["dl"], lw=2, label=f"Deep learning (AUPRC {results.loc['Deep learning', 'AUPRC']:.3f})")
ax.scatter(results.loc["Deep learning", "Recall"], results.loc["Deep learning", "Precision"], c=C["dl"], s=70, edgecolor="k", zorder=5)
ax.scatter(results.loc["Symbolic rule", "Recall"], results.loc["Symbolic rule", "Precision"], marker="D", s=80, c=C["symbolic"], edgecolor="k", label="Symbolic rule", zorder=5)
ax.axhline(y_test.mean(), c="k", ls=":", label=f"no-skill ({y_test.mean():.1%})")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0.4, 1.02); ax.set_title("Precision-recall curve, test set"); ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); savefig("11_precision_recall_curve")

### Model comparison
**What it represents.** Headline metrics for each approach. Left: the full test set with 95% patient-level bootstrap intervals. Right: the 40-image subset a human labelled, where all approaches are compared on identical images (no intervals: n is too small for them to mean much).
**How to read it.** If two bars' intervals overlap heavily the difference is not established. Look at recall and specificity separately: a method can reach a decent F1 simply by saying "pneumonia" a lot.

In [ ]:
mets, labs = ["F1", "Recall", "Specificity", "BalAcc"], ["F1", "Recall", "Specificity", "Balanced acc."]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [3, 2.6]})
apps = ["Symbolic rule", "Deep learning"]; w = 0.34
for k, a in enumerate(apps):
    v = np.array([results.loc[a, m] for m in mets])
    err = np.array([v - [ci_lo.loc[a, m] for m in mets], [ci_hi.loc[a, m] for m in mets] - v])
    a1.bar(np.arange(4) + (k - 0.5) * w, v, w, yerr=err, capsize=3, color=COL[a], label=a)
a1.set_xticks(range(4), labs); a1.set_ylim(0, 1.08); a1.set_title(f"Full test set (n = {len(y_test)}), 95% CI")
a1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=8, frameon=False)
if subset_res:
    apps2, w2 = ["Human", "Symbolic rule", "Deep learning"], 0.26
    for k, a in enumerate(apps2): a2.bar(np.arange(4) + (k - 1) * w2, [subset_res[a][m] for m in mets], w2, color=COL[a], label=a)
    a2.set_xticks(range(4), labs); a2.set_ylim(0, 1.08); a2.set_title(f"Human-baseline subset (n = {mask.sum()})")
    a2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=8, frameon=False)
else:
    a2.text(0.5, 0.5, "subset comparison not available\nin this run", ha="center", transform=a2.transAxes); a2.axis("off")
plt.tight_layout(); savefig("12_model_comparison")

### Confidence intervals (forest plot)
**What it represents.** One panel per metric: point estimate plus 95% patient-level bootstrap interval for each approach, including the trivial *always positive* baseline.
**Why it matters.** It shows which gaps are meaningful with only 624 test images in clusters. Intervals that overlap cannot be told apart. Note how *always positive* has a competitive F1 but zero MCC and zero specificity.

In [ ]:
fm = ["F1", "BalAcc", "Specificity", "Recall", "MCC"]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2), sharey=True)
for ax, m in zip(axes, fm):
    for i, a in enumerate(APPS):
        v = results.loc[a, m]
        ax.errorbar(v, i, xerr=[[v - ci_lo.loc[a, m]], [ci_hi.loc[a, m] - v]], fmt="o", c=COL[a], capsize=4, ms=7)
        ax.text(v, i + 0.28, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_title(m); ax.set_yticks(range(len(APPS)), APPS); ax.invert_yaxis()
plt.suptitle("Test metrics with 95% patient-level bootstrap CIs", y=1.03)
plt.tight_layout(); savefig("13_metric_confidence_intervals")

### Score distributions
**What it represents.** The model's predicted pneumonia probability for NORMAL vs PNEUMONIA images, on validation (left) and test (right), with the chosen threshold.
**How to read it.** Two well-separated humps mean an easy problem; the overlap is where errors live. Comparing validation and test shows whether the scores shift between the sets, which is why a threshold must not be assumed to transfer blindly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
bins = np.linspace(0, 1, 26)
for ax, (nm, yy, pp) in zip(axes, [("validation", yv, p_val), ("test", y_test, p_test)]):
    ax.hist(pp[yy == 0], bins, alpha=0.65, color=C["normal"], density=True, label="NORMAL")
    ax.hist(pp[yy == 1], bins, alpha=0.65, color=C["pneu"], density=True, label="PNEUMONIA")
    ax.axvline(THR, c="k", ls="--", label=f"threshold {THR:.2f}"); ax.set_title(f"Predicted scores, {nm}"); ax.set_xlabel("predicted P(pneumonia)")
axes[0].set_ylabel("density"); axes[0].legend()
plt.tight_layout(); savefig("14_score_distributions")

### Threshold sweep, validation vs test
**What it represents.** Recall, specificity and F1 as functions of the decision threshold, on validation (dashed) and test (solid).
**How to read it.** Where the dashed and solid curves of one colour differ, validation-based choices do not carry over to test: that is the distribution shift (different prevalence; the Kaggle test folder is also known to be harder). It also makes the recall-vs-specificity trade-off explicit.

In [ ]:
ts = np.linspace(0.01, 0.99, 99)
fig, ax = plt.subplots(figsize=(8, 4.6))
for (nm, yy, pp, ls) in [("val", yv, p_val, "--"), ("test", y_test, p_test, "-")]:
    ms = [binary_metrics(yy, pp >= t) for t in ts]
    for key, colr in (("Recall", C["pneu"]), ("Specificity", C["normal"]), ("F1", "#2ca02c")):
        ax.plot(ts, [m[key] for m in ms], c=colr, ls=ls, label=f"{key} ({nm})")
ax.axvline(THR, c="k", ls=":", label="chosen threshold")
ax.set_xlabel("threshold"); ax.set_ylabel("metric"); ax.set_title("Threshold sweep"); ax.legend(ncol=2, fontsize=8)
plt.tight_layout(); savefig("15_threshold_sweep")

### Calibration (reliability diagram)
**What it represents.** Left: among images given probability around *p*, what fraction really were pneumonia? A perfectly calibrated model lies on the diagonal. Right: how many test images fall in each probability range. The legend gives the Brier score (mean squared error of the probabilities) and ECE.
**How to read it.** Points below the diagonal = over-confident. Networks are typically over-confident, and class weights deliberately shift probabilities. Good ranking (AUROC) and good calibration are different things; the long-training figure showed validation loss (which punishes over-confidence) can worsen while AUROC does not.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
frac, mean_p = calibration_curve(y_test, p_test, n_bins=10, strategy="uniform")
axes[0].plot(mean_p, frac, "o-", c=C["dl"], label=f"Deep learning (Brier {results.loc['Deep learning', 'Brier']:.3f}, ECE {results.loc['Deep learning', 'ECE']:.3f})")
axes[0].plot([0, 1], [0, 1], "k:", label="perfect calibration")
axes[0].set_xlabel("mean predicted probability"); axes[0].set_ylabel("observed fraction pneumonia"); axes[0].set_title("Reliability diagram (test)"); axes[0].legend(fontsize=8)
axes[1].hist(p_test, np.linspace(0, 1, 21), color=C["dl"], alpha=0.7); axes[1].set_xlabel("predicted probability"); axes[1].set_ylabel("test images"); axes[1].set_title("Prediction histogram")
plt.tight_layout(); savefig("16_calibration")

## 10. Figures: the three course considerations

### Explainability: Grad-CAM
**What it represents.** A heat-map over the X-ray showing which regions pushed the model's score towards "pneumonia" (Gradient-weighted Class Activation Mapping: the last convolutional feature maps, weighted by how strongly the pneumonia output's gradient flows through each). Rows: the most confident **true positive**, **true negative**, **false positive** and **false negative** (two examples each: raw image, then overlay).
**Why it matters.** It checks that the model looks at *lung tissue*. This dataset is known for shortcut risks (corner markers, borders, scanner differences, text); a model attending to those would be right for the wrong reason. Heat on the lungs for true positives is the healthy pattern; also look at what the false positives light up. Unlike a rule-based method, a neural network cannot state a feature that drove its answer, so this is the best available explanation, and it is a sanity check, not a proof.

In [ ]:
final_model = keras.saving.load_model(FINAL_PATH, compile=False)
_base = get_backbone(final_model)
_bi = final_model.layers.index(_base)
_front = final_model.layers[1:_bi]                                        # to_rgb, augment (inactive), rescaling
_last4d = [l for l in _base.layers if len(l.output.shape) == 4][-1]       # last convolutional feature map
_A = keras.Model(_base.inputs, _last4d.output)
_tail = _base.layers[_base.layers.index(_last4d) + 1:]                    # pooling (and norm) after it
_dense = final_model.get_layer("pneumonia")


def grad_cam(img):
    z = tf.constant(img[None, ..., None].astype("float32"))
    with tf.GradientTape() as tape:
        for l in _front: z = l(z, training=False)
        conv = _A(z, training=False); tape.watch(conv)
        f = conv
        for l in _tail: f = l(f)
        pred_ = _dense(f)
    g = tf.cast(tape.gradient(pred_, conv), tf.float32); conv32 = tf.cast(conv, tf.float32)
    cam = tf.nn.relu(tf.reduce_sum(tf.reduce_mean(g, axis=(1, 2), keepdims=True) * conv32, axis=-1, keepdims=True))
    cam = tf.image.resize(cam, (IMG, IMG))[0, ..., 0].numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)


Xte_all = X[idx["test"]]
cats = [("True positive", (y_test == 1) & (pred_dl == 1), True), ("True negative", (y_test == 0) & (pred_dl == 0), False),
        ("False positive", (y_test == 0) & (pred_dl == 1), True), ("False negative", (y_test == 1) & (pred_dl == 0), False)]
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for r, (title, msk, desc) in enumerate(cats):
    sel = np.where(msk)[0]
    sel = sel[np.argsort(-p_test[sel] if desc else p_test[sel])][:2]       # most confident examples of the category
    for k in range(2):
        a_img, a_cam = axes[r, 2 * k], axes[r, 2 * k + 1]; a_img.axis("off"); a_cam.axis("off")
        if k < len(sel):
            i = sel[k]; img = Xte_all[i]
            a_img.imshow(img, cmap="gray", vmin=0, vmax=255); a_img.set_title(f"{title}\n{test_df['Example ID'][i]}  ({test_df.subtype[i]})", fontsize=9)
            a_cam.imshow(img, cmap="gray", vmin=0, vmax=255); a_cam.imshow(grad_cam(img), cmap="jet", alpha=0.45, extent=(0, IMG, IMG, 0))
            a_cam.set_title(f"Grad-CAM, P(pneumonia) = {p_test[i]:.2f}", fontsize=9)
plt.tight_layout(); savefig("17_gradcam")

### Error gallery
**What it represents.** The most confident mistakes: top row false positives (normal images scored as pneumonia), bottom row false negatives (pneumonia scored as normal, with their subtype).
**Why it matters.** Qualitative error analysis: look for patterns (faint viral pneumonia, unusual exposure, odd framing, devices in frame) that suggest what the model cannot do. The course asks you to describe the types of inputs associated with errors.

In [ ]:
fp = np.where((y_test == 0) & (pred_dl == 1))[0]; fp = fp[np.argsort(-p_test[fp])][:4]
fn = np.where((y_test == 1) & (pred_dl == 0))[0]; fn = fn[np.argsort(p_test[fn])][:4]
fig, axes = plt.subplots(2, 4, figsize=(12, 6.6))
for row, (ix, kind) in enumerate([(fp, "False positive"), (fn, "False negative")]):
    for k in range(4):
        ax = axes[row, k]; ax.axis("off")
        if k < len(ix):
            i = ix[k]; ax.imshow(Xte_all[i], cmap="gray")
            ax.set_title(f"{kind}: {test_df['Example ID'][i]}\n{test_df.subtype[i]}, P = {p_test[i]:.2f}", fontsize=9, color=C["pneu"] if kind.startswith("False p") else C["normal"])
plt.tight_layout(); savefig("18_error_gallery")

### Fairness: bacterial vs viral pneumonia
**What it represents.** How often each approach is right within each test subgroup: NORMAL images (this is specificity), bacterial pneumonia (recall) and viral pneumonia (recall). Counts are in the axis labels.
**Why it matters.** One overall recall hides that the two pneumonia types look different on X-ray (bacterial typically shows a dense lobar patch, viral a more diffuse pattern). It is the only subgroup readable from the data; all images come from **one hospital and one age group**, so other groups (age, sex, scanner) cannot be checked, which is an important limitation to state.

In [ ]:
groups = [("normal", "NORMAL\n(specificity)"), ("bacterial", "Bacterial\n(recall)"), ("viral", "Viral\n(recall)")]
fig, ax = plt.subplots(figsize=(9, 4.4)); w = 0.34
sub_rows = []
for k, a in enumerate(["Symbolic rule", "Deep learning"]):
    vals = []
    for g, _ in groups:
        mk = (test_df.subtype == g).values
        vals.append(((pred[a][mk] == 0) if g == "normal" else (pred[a][mk] == 1)).mean() if mk.any() else np.nan)
        sub_rows.append(dict(approach=a, group=g, n=int(mk.sum()), correct_rate=vals[-1]))
    bars = ax.bar(np.arange(3) + (k - 0.5) * w, vals, w, color=COL[a], label=a)
    for b, v in zip(bars, vals): ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(range(3), [f"{lab}\nn = {(test_df.subtype == g).sum()}" for g, lab in groups]); ax.set_ylim(0, 1.1)
ax.set_title("Correct-rate per test subgroup"); ax.legend(loc="lower right", fontsize=8)
pd.DataFrame(sub_rows).to_csv(RES_DIR / "subgroup_performance.csv", index=False)
plt.tight_layout(); savefig("19_subgroup_performance")

### Embedding map (t-SNE)
**What it represents.** The numbers the final model computes just before its classifier (the pooled backbone features), squeezed into 2-D with t-SNE, for all validation and test images. Left: coloured by true class (circles = val, triangles = test). Right: coloured by NORMAL / bacterial / viral.
**How to read it.** Clusters mean the network has organised the images by diagnosis. Overlap regions are where it is unsure; viral cases sitting between normal and bacterial would match the subgroup result. Test points that sit apart from validation points of the same class indicate domain shift. (t-SNE distances are only meaningful locally.)

In [ ]:
feat_model = keras.Model(final_model.inputs, final_model.get_layer("head_dropout").input)
emb_ix = np.concatenate([idx["val"], idx["test"]])
emb = feat_model.predict(X[emb_ix][..., None], batch_size=64, verbose=0)
xy = TSNE(n_components=2, init="pca", perplexity=min(30, (len(emb) - 1) / 3), random_state=SEED).fit_transform(emb)
is_test = np.r_[np.zeros(len(idx["val"]), bool), np.ones(len(idx["test"]), bool)]
lab, sub_lab = y[emb_ix].astype(int), df.subtype.values[emb_ix]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
for s, mk_, nm in ((False, "o", "val"), (True, "^", "test")):
    for c, cn in ((0, "NORMAL"), (1, "PNEUMONIA")):
        mk = (is_test == s) & (lab == c)
        axes[0].scatter(xy[mk, 0], xy[mk, 1], s=14, marker=mk_, c=C["normal"] if c == 0 else C["pneu"], alpha=0.6, label=f"{cn} ({nm})")
for g, colr in (("normal", C["normal"]), ("bacterial", "#E8A87C"), ("viral", "#8B0000")):
    mk = sub_lab == g; axes[1].scatter(xy[mk, 0], xy[mk, 1], s=14, c=colr, alpha=0.6, label=g)
axes[0].set_title("t-SNE of final-model features: class and split"); axes[1].set_title("same map: pneumonia subtype")
for a in axes: a.legend(fontsize=8, markerscale=1.5); a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); savefig("20_embedding_tsne")

### Cost and capacity (and sustainability)
**What it represents.** Left: validation AUROC against **time to classify one image** for each backbone (bubble size = number of parameters; the scratch CNN is included as a reference). Middle: model size on disk. Right: total **training minutes** spent in each stage of this notebook.
**Why it matters.** The course asks what each approach costs in development time and compute, including time per prediction. A model that is slightly more accurate but several times slower or larger may not be the better choice, and total training time is a rough proxy for energy use. Timings depend on the hardware (shown in the title), so compare them with each other, not with other machines.

In [ ]:
def measure_cost(cfg):
    keras.backend.clear_session()
    m = build_model({**cfg, "pretrained": False})        # weights do not affect speed or size
    xs = X[:64][..., None]
    # Warm up with the SAME shapes that are timed (the first call of each shape builds / compiles the graph, which
    # used to be counted as inference time), then time the median of repeated calls.
    m.predict(xs, batch_size=BATCH, verbose=0)
    batch = []
    for _ in range(3):
        t0 = time.time(); m.predict(xs, batch_size=BATCH, verbose=0); batch.append(time.time() - t0)
    batch_ms = float(np.median(batch)) / len(xs) * 1000
    infer = tf.function(lambda t: m(t, training=False))        # compiled graph, as predict() uses (eager calls are far slower)
    one = xs[:1].astype("float32")
    for _ in range(3):
        infer(one).numpy()                                      # warm-up (also forces the GPU to finish)
    single = []
    for _ in range(20):
        t0 = time.time(); infer(one).numpy(); single.append((time.time() - t0) * 1000)
    tmp = WORK / "size_probe.keras"; m.save(tmp); size = tmp.stat().st_size / 1e6; tmp.unlink()
    return dict(params=m.count_params(), ms_per_image_batch=batch_ms, ms_single_image=float(np.median(single)), size_mb=size)


cost_rows = []
for bb in P["backbones"]:
    c = measure_cost({**DEFAULT, "backbone": bb}); c.update(model=bb, val_auroc=s2_runs[bb]["best_auc"], sec_per_epoch=s2_runs[bb]["history"].seconds.mean()); cost_rows.append(c)
if scratch_run:
    c = measure_cost({**DEFAULT, "backbone": "scratch"}); c.update(model="scratch CNN (reference)", val_auroc=scratch_run["best_auc"], sec_per_epoch=scratch_run["history"].seconds.mean()); cost_rows.append(c)
cost = pd.DataFrame(cost_rows)[["model", "params", "val_auroc", "ms_per_image_batch", "ms_single_image", "size_mb", "sec_per_epoch"]]
cost.to_csv(RES_DIR / "cost_and_capacity.csv", index=False)
stage_min = (pd.DataFrame(ALL_RUNS).drop_duplicates("id").groupby("stage").seconds.sum() / 60).rename("minutes")
stage_min.to_csv(RES_DIR / "training_minutes_by_stage.csv")
display(cost.style.format({"params": "{:,}", "val_auroc": "{:.4f}", "ms_per_image_batch": "{:.1f}", "ms_single_image": "{:.1f}", "size_mb": "{:.1f}", "sec_per_epoch": "{:.1f}"}))

fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
for _, r in cost.iterrows():
    axs[0].scatter(r.ms_per_image_batch, r.val_auroc, s=60 + r.params / 1e5, alpha=0.7, edgecolor="k")
    axs[0].annotate(r.model, (r.ms_per_image_batch, r.val_auroc), textcoords="offset points", xytext=(6, 6), fontsize=8)
axs[0].set_xlabel("ms per image (batched inference)"); axs[0].set_ylabel("validation AUROC"); axs[0].set_title("Accuracy vs speed (bubble = parameters)")
axs[1].barh(cost.model, cost.size_mb, color="#9aa5b1"); axs[1].set_xlabel("MB on disk (.keras)"); axs[1].set_title("Model size")
axs[2].barh(stage_min.index[::-1], stage_min.values[::-1], color="#9aa5b1"); axs[2].set_xlabel("minutes"); axs[2].set_title("Training time spent per stage")
fig.suptitle(f"Cost and capacity (hardware: {DEVICE})", y=1.02)
plt.tight_layout(); savefig("21_cost_and_capacity")

## 11. Experiments that explain *why* it works (validation only)

### Learning curve (data efficiency)
**What it represents.** Validation AUROC when the final recipe is trained on 10%, 25%, 50% and 100% of the training patients (same epoch budget each), against the scratch CNN reference.
**How to read it.** A curve still rising at 100% means more data would help (data-limited); a flat one means the model is the limit. A big gap at small fractions is the benefit of transfer learning: pretrained features substitute for data. Subsets are drawn by *patient*, so one child's images stay together.

In [ ]:
STAGE = "5 learning curve"


def subsample_train(frac, seed=SEED):
    tr = idx["train"]
    if frac >= 1: return tr
    g = df.patient_id.values[tr]; ug = np.unique(g)
    np.random.default_rng(seed).shuffle(ug)
    keep = set(ug[:max(1, int(len(ug) * frac))])
    return tr[[x in keep for x in g]]


lc = []
final_recipe = {**best_cfg, "epochs": P["lc_epochs"]}
scratch_recipe = {**DEFAULT, "backbone": "scratch", "pretrained": False, "unfreeze": 1.0, "lr": 1e-3, "head_epochs": 0, "epochs": P["lc_epochs"] + P["head_epochs"]}
for frac in P["lc_fracs"] + [1.0]:
    ix = subsample_train(frac)
    recipes = [("Pretrained + fine-tuned (final recipe)", final_recipe)] + ([("Scratch CNN (reference)", scratch_recipe)] if P["include_scratch"] else [])
    for nm, cfg in recipes:
        r = cached_run(cfg, train_ix=ix, verbose=0)
        lc.append(dict(model=nm, frac=frac, n_images=len(ix), val_auroc=r["best_auc"]))
        print(f"{nm:<40} {frac:>4.0%} of train ({len(ix):>5} images): val AUROC {r['best_auc']:.4f}")
lc = pd.DataFrame(lc); lc.to_csv(RES_DIR / "learning_curve.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4.6))
for nm, g in lc.groupby("model"):
    ax.plot(g.n_images, g.val_auroc, "o-", lw=2, c=C["dl"] if nm.startswith("Pre") else C["scratch"], label=nm)
ax.set_xscale("log"); ax.set_xlabel("training images (log scale)"); ax.set_ylabel("validation AUROC"); ax.set_title("Learning curve"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("22_learning_curve")

### Ablation study
**What it represents.** The final recipe with one ingredient changed at a time (same epoch budget): no pretraining (random weights; every layer trained, BatchNorm statistics adapted quickly), no augmentation, no class weighting, adding horizontal flips, or all backbone layers unfrozen (the final recipe trains the head only). Bars are validation AUROC.
**How to read it.** A big drop when an ingredient is removed means it earns its place; no change means it was not needed. These are single-seed runs, so differences under about 0.01 AUROC are inside the run-to-run noise seen in the long-training seeds. Ablations use *validation* AUROC so the test set stays untouched.

In [ ]:
STAGE = "6 ablations"
base_ab = {**best_cfg, "epochs": P["ablation_epochs"]}
variants = {"Full recipe": base_ab,
            "No pretraining (random init)": {**base_ab, "pretrained": False, "unfreeze": 1.0, "lr": 1e-3, "train_bn": True, "bn_momentum": 0.9},
            "No augmentation": {**base_ab, "aug": "none"},
            "No class weighting": {**base_ab, "class_weight": False},
            "+ Horizontal flip": {**base_ab, "hflip": True},
            "All layers unfrozen": {**base_ab, "unfreeze": 1.0}}
abl = []
for nm, cfg in variants.items():
    r = cached_run(cfg, verbose=0); abl.append(dict(variant=nm, val_auroc=r["best_auc"])); print(f"{nm:<30} val AUROC {r['best_auc']:.4f}")
abl = pd.DataFrame(abl); abl.to_csv(RES_DIR / "ablation.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(abl.variant[::-1], abl.val_auroc[::-1], color=[C["dl"] if v == "Full recipe" else "#9aa5b1" for v in abl.variant[::-1]])
for i, v in enumerate(abl.val_auroc[::-1]): ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(max(0.4, abl.val_auroc.min() - 0.05), min(1.02, abl.val_auroc.max() + 0.03))
ax.set_xlabel("validation AUROC"); ax.set_title("Ablation: one change at a time")
plt.tight_layout(); savefig("23_ablation")

## 12. Summary and downloads

In [ ]:
(RES_DIR / "run_info.json").write_text(json.dumps(dict(
    profile=PROFILE, device=DEVICE, tensorflow=tf.__version__, keras=keras.__version__, seed=SEED, image_size=IMG,
    threshold=THR, threshold_rule="Youden's J on validation", final_config=long_cfg, final_seed=final["seed"],
    final_best_epoch=final["best_epoch"], total_epochs_final_run=len(final["history"])), indent=2, default=str))

print("Headline (test set, patient-level bootstrap 95% CI):")
display(fmt[["F1", "Recall", "Specificity", "BalAcc", "MCC", "AUROC"]])
print("Files written:")
for f in SAVED: print("  ", (OUT / f).relative_to(PROJECT) if OUT.is_relative_to(PROJECT) else OUT / f)
for f in sorted(RES_DIR.glob("*")): print("  ", f.relative_to(PROJECT) if f.is_relative_to(PROJECT) else f)
print("  ", FINAL_PATH.relative_to(PROJECT) if FINAL_PATH.is_relative_to(PROJECT) else FINAL_PATH)

if MODE == "full":
    print("\n--- eval/eval.py (the shared scorer) ---")
    print(subprocess.run([sys.executable, str(PROJECT / "eval" / "eval.py")], capture_output=True, text=True).stdout)

if IN_COLAB and MODE == "full":
    bundle = Path("/content/dl_bundle"); shutil.rmtree(bundle, ignore_errors=True)
    shutil.copytree(FIG_DIR, bundle / "approaches/3_deep_learning/figures"); shutil.copytree(RES_DIR, bundle / "approaches/3_deep_learning/results")
    (bundle / "approaches/3_deep_learning/models").mkdir(parents=True); shutil.copy(FINAL_PATH, bundle / "approaches/3_deep_learning/models")
    (bundle / "eval/predictions").mkdir(parents=True); shutil.copy(PROJECT / "eval/predictions/deep_learning_eval.csv", bundle / "eval/predictions")
    zp = shutil.make_archive("/content/dl_outputs", "zip", bundle)
    from google.colab import files
    files.download(zp)      # unzip into your repo root: the folders already match the repo layout

**Limitations to state in the write-up**
- **Possible train/test overlap.** About 170 pneumonia patient numbers appear in both Kaggle's train and test folders (`data/data_checks.txt`). File names cannot show whether these are the same children, so test scores may be optimistic.
- **Normal X-rays have no patient ID**, so one child's normal images can sit on both sides of the train/val split; validation is slightly optimistic and the tuned threshold slightly biased.
- **One hospital, one age group**, and a single validation hold-out of about 1,000 images: small differences between configurations are noise, which is why close calls were re-run with seeds and ties go to the simpler model.
- **ImageNet pretraining** uses photographs, not medical images. State this when comparing with the other approaches.
- **Distribution shift:** test prevalence (62.5%) differs from train/val (74%). The prevalence-independent threshold rule reduces, but does not remove, the effect.
- Grad-CAM shows where the model looked, not a proof of *why*.
- Timings in the cost figure depend on the hardware they were measured on.